# Eval-only rerun for the LFM2.5-230M SAEs

Loads the trained SAEs from the `lfm25-sae-training` kernel output (mounted
under `/kaggle/input/`) and re-evaluates them with special-token positions
excluded, matching training (`exclude_special_tokens=True`).

In [ ]:
%pip install -q -U sae-lens "transformers>=5.4"

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from sae_lens import SAE
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "LiquidAI/LFM2.5-230M"
import glob

_hits = glob.glob("/kaggle/input/**/sae_layer6/cfg.json", recursive=True)
if not _hits:
    import subprocess

    print(subprocess.run(["find", "/kaggle/input", "-maxdepth", "3"], capture_output=True, text=True).stdout)
    raise FileNotFoundError("sae_layer6/cfg.json not found under /kaggle/input")
SAE_ROOT = Path(_hits[0]).parent.parent
print("SAE_ROOT =", SAE_ROOT)
LAYERS = [6, 10]
CONTEXT_SIZE = 1024
EVAL_BATCHES = 16
EVAL_BS = 8
WORK = "/kaggle/working"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16)
    .to("cuda:0")
    .eval()
)

ds = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)
ds = ds.shuffle(seed=1234, buffer_size=1000)
texts = []
it = iter(ds)
while len(texts) < EVAL_BATCHES * EVAL_BS:
    t = next(it)["text"]
    if len(t) > 2000:
        texts.append(t)

special = torch.tensor(tok.all_special_ids, device="cuda:0")
eval_summary = {}
for layer in LAYERS:
    sae = SAE.load_from_disk(str(SAE_ROOT / f"sae_layer{layer}"), device="cuda:0")
    captured = {}

    def hook(_m, _i, output):
        captured["resid"] = output[0] if isinstance(output, tuple) else output

    handle = model.model.layers[layer].register_forward_hook(hook)

    l0s, evs, evs_with_special = [], [], []
    bos_err_ratio = []
    fired = torch.zeros(sae.cfg.d_sae, dtype=torch.bool, device="cuda:0")
    rng = np.random.default_rng(0)
    track_feats = rng.choice(sae.cfg.d_sae, size=6, replace=False)
    top_store = {int(f): [] for f in track_feats}

    with torch.no_grad():
        for b in range(EVAL_BATCHES):
            batch = texts[b * EVAL_BS : (b + 1) * EVAL_BS]
            enc = tok(
                batch,
                return_tensors="pt",
                truncation=True,
                max_length=CONTEXT_SIZE,
                padding="max_length",
            ).to("cuda:0")
            model(**enc)
            acts = captured["resid"].float()
            attn = enc["attention_mask"].bool()
            not_special = ~torch.isin(enc["input_ids"], special)

            def ev(m):
                x = acts[m]
                recon = sae.decode(sae.encode(x))
                return (1 - (x - recon).pow(2).sum() / (x - x.mean(0)).pow(2).sum()).item()

            evs.append(ev(attn & not_special))
            evs_with_special.append(ev(attn))

            x = acts[attn & not_special]
            ids = enc["input_ids"][attn & not_special]
            feats = sae.encode(x)
            l0s.append((feats > 0).float().sum(-1).mean().item())
            fired |= (feats > 0).any(0)

            x_special = acts[attn & ~not_special]
            if len(x_special) > 0:
                r_s = sae.decode(sae.encode(x_special))
                bos_err_ratio.append(
                    ((x_special - r_s).pow(2).sum() / (x - sae.decode(feats)).pow(2).sum()).item()
                )

            for f in track_feats:
                vals = feats[:, int(f)]
                top = torch.topk(vals, k=min(10, len(vals)))
                top_store[int(f)].extend(
                    (v.item(), ids[i].item())
                    for v, i in zip(top.values, top.indices)
                    if v.item() > 0
                )

    handle.remove()
    summary = {
        "hook": f"model.layers.{layer}",
        "L0": round(float(np.mean(l0s)), 2),
        "explained_variance": round(float(np.mean(evs)), 4),
        "explained_variance_incl_special": round(float(np.mean(evs_with_special)), 4),
        "special_vs_normal_sq_err_ratio": round(float(np.mean(bos_err_ratio)), 2),
        "dead_feature_fraction_eval": round(1 - fired.float().mean().item(), 4),
    }
    eval_summary[f"layer{layer}"] = summary
    print(f"\n===== layer {layer}: {summary}")
    for f, entries in top_store.items():
        entries.sort(reverse=True)
        toks = [f"{tok.decode([tid])!r}:{v:.2f}" for v, tid in entries[:8]]
        print(f"  feature {f:5d} top tokens: {', '.join(toks)}")

Path(WORK, "eval_summary_fixed.json").write_text(json.dumps(eval_summary, indent=2))
print("\nwrote", Path(WORK, "eval_summary_fixed.json"))